# **Project Name**    - Netflix Movies and TV Shows Clustering


##### **Project Type**    - Unsupervised Clustering
##### **Contribution**    - Individual
##### **Team Member 1 -** Biplab Mondal

# **Project Summary -**

Goal : To analyze and cluster Netflix’s catalog (sourced from Flixable) to improve user experience and reduce subscriber churn.

Key Details:

1. Dataset:
    * Contains 7,787 records and 11 attributes (movies and TV shows till 2019).
2. Objective:
    * Group similar shows/movies into clusters.
    * Discover insights and trends in Netflix content.
3. Data Preparation:
    * Handled missing values.
    * Performed Exploratory Data Analysis (EDA) for better understanding.
4. Features Used for Clustering:
    * Cast, Country, Genre, Director, Rating, and Description.
    * Applied TF-IDF Vectorizer for text processing and tokenization.
5. Dimensionality Reduction:
    * Used Principal Component Analysis (PCA) to reduce high dimensional data.
6. Clustering Methods:
    * K-Means Clustering and Agglomerative Hierarchical Clustering.
    * Determined optimal clusters using Elbow Method, Silhouette Score, and Dendrogram.
7. Recommendation System:
    * Built a content-based recommender using Cosine Similarity.
    * Provides 10 similar show recommendations based on user’s watched show.


# **Problem Statement**


Netflix, a streaming service, offers a wide range of TV shows and movies for viewers to watch at their convenience. With a monthly subscription, users can access a vast library of content, including original series and films. The platform allows users to create multiple profiles, making it easy for family or roommates to have personalised viewing experiences. Netflix also lets users download content for offline viewing, making it great for frequent travellers or those with limited internet access. As of 2022-Q2, over 220 million people had signed up for Netflix’s online streaming service, making it the largest OTT provider worldwide. To improve the user experience and reduce churn, Netflix must efficiently cluster the shows on its platform. By creating clusters, we can understand similar and different shows and provide personalised recommendations. This project aims to classify and group Netflix shows into specific clusters, ensuring shows in the same cluster are similar and those in different clusters are different.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [3]:
# Import Libraries and modules

# libraries that are used for analysis and visualization
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# Word Cloud library
# from wordcloud import WordCloud, STOPWORDS

# libraries used to process textual data
import string
string.punctuation
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

# libraries used to implement clusters
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering
import scipy.cluster.hierarchy as shc

# libraries that are used to construct a recommendation system
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Library of warnings would assist in ignoring warnings issued
import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/biplab/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


### Dataset Loading

In [4]:
# Load Dataset

netflix_df = pd.read_csv('/Users/biplab/Downloads/Netflix/NETFLIX MOVIES AND TV SHOWS CLUSTERING.csv')

### Dataset First View

In [5]:
# Dataset First Look

netflix_df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."
3,s4,Movie,9,Shane Acker,"Elijah Wood, John C. Reilly, Jennifer Connelly...",United States,"November 16, 2017",2009,PG-13,80 min,"Action & Adventure, Independent Movies, Sci-Fi...","In a postapocalyptic world, rag-doll robots hi..."
4,s5,Movie,21,Robert Luketic,"Jim Sturgess, Kevin Spacey, Kate Bosworth, Aar...",United States,"January 1, 2020",2008,PG-13,123 min,Dramas,A brilliant group of students become card-coun...


### Dataset Rows & Columns count

In [6]:
# Dataset Rows & Columns count

netflix_df.shape

(7787, 12)

### Dataset Information

In [7]:
# Dataset Info

netflix_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7787 entries, 0 to 7786
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       7787 non-null   object
 1   type          7787 non-null   object
 2   title         7787 non-null   object
 3   director      5398 non-null   object
 4   cast          7069 non-null   object
 5   country       7280 non-null   object
 6   date_added    7777 non-null   object
 7   release_year  7787 non-null   int64 
 8   rating        7780 non-null   object
 9   duration      7787 non-null   object
 10  listed_in     7787 non-null   object
 11  description   7787 non-null   object
dtypes: int64(1), object(11)
memory usage: 730.2+ KB


#### Duplicate Values

In [8]:
# Dataset Duplicate Value Count

value = len(netflix_df[netflix_df.duplicated()])
print("Duplicate values in the data set = ",value)

Duplicate values in the data set =  0


#### Missing Values/Null Values

In [9]:
# Missing Values/Null Values Count

print(netflix_df.isnull().sum())

show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64


In [10]:
# Visualizing the missing values

round(netflix_df.isna().sum()/len(netflix_df)*100, 2)

show_id          0.00
type             0.00
title            0.00
director        30.68
cast             9.22
country          6.51
date_added       0.13
release_year     0.00
rating           0.09
duration         0.00
listed_in        0.00
description      0.00
dtype: float64

### What did you know about your dataset?

The dataset, sourced from the online streaming industry, requires analysis, clustering methods, and a content-based recommendation system. Clustering, a machine learning technique, groups similar data points. Clustering algorithms identify these groups in a dataset, without prior knowledge. The dataset has 7787 rows and 12 columns, with missing values in director, cast, country, date_added, and rating. It lacks duplicates. Since each row represents a movie, imputing null values is impossible. Due to the small dataset, we don’t want to lose data. After analysing each column, we impute numeric values with empty strings.

## ***2. Understanding Your Variables***

In [11]:
# Dataset Columns
netflix_df.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')

In [16]:
# Dataset Describe
netflix_df.describe()

,release_year
count,7787.000000
mean,2013.932580
std,8.757395
min,1925.000000
25%,2013.000000
50%,2017.000000
75%,2018.000000
max,2021.000000


### Variables Description

show_id : Unique ID for every Movie/Show

type : Identifier - Movie/Show

title : Title of the Movie/Show

director : Director of the Movie/Show

cast : Actors involved in the Movie/Show

country : Country where the Movie/Show was produced

date_added : Date it was added on Netflix

release_year : Actual Release year of the Movie/Show

rating : TV Rating of the Movie/Show

duration : Total Duration - in minutes or number of seasons

listed_in : Genre

description : The Summary description

### Check Unique Values for each variable.

In [17]:
# Check Unique Values for each variable.

for i in netflix_df.columns.tolist():
  print("No. of unique values in",i,"is",netflix_df[i].nunique())
     


No. of unique values in show_id is 7787
No. of unique values in type is 2
No. of unique values in title is 7787
No. of unique values in director is 4049
No. of unique values in cast is 6831
No. of unique values in country is 681
No. of unique values in date_added is 1565
No. of unique values in release_year is 73
No. of unique values in rating is 14
No. of unique values in duration is 216
No. of unique values in listed_in is 492
No. of unique values in description is 7769
